# QuAM — Query Answering Machine for 3D Meshes
**Course:** 15-288 Spring 2026 · **Deliverable:** D3 — Model Iterations

Three iterations of ML models for 15-class fruit shape classification from raw 3D mesh files.

| Iteration | Representation | Model | Input Shape |
|-----------|---------------|-------|-------------|
| D3.3 | Voxel Grid (3D CNN) | SVM on PCA-projected voxels | 32³ → R^k |
| D3.4 | Point Cloud (PointNet-style) | SVM on geometric descriptors | 1024 × 3 → R^29 |
| D3.5 | Mesh Graph (GCN-style) | SVM on graph statistics | G(V,E) → R^16 |

## D3.0 — Setup

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import KDTree
from scipy.stats import skew as scipy_skew
from sklearn.decomposition import PCA
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix
from sklearn.model_selection import LeaveOneOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings('ignore')

for _candidate in [Path('../data'), Path('data')]:
    if _candidate.exists():
        DATA_DIR = _candidate
        break
else:
    raise FileNotFoundError('Cannot find data/ directory. Run from notebooks/ or repo root.')

print(f'Data directory: {DATA_DIR.resolve()}')

## D3.1 — Dataset Loading

Each processed sample lives in `data/<session_id>/` and contains:
- `voxels.json` — 32³ occupancy matrix
- `points.json` — 1024-point surface sample
- `graph.json` — mesh graph (nodes + edges)

The ground-truth class label is parsed from the original filename using longest-match-first against the 15 known fruit names.

In [ ]:
CLASS_NAMES = [
    'apple', 'avocado', 'banana', 'cherry', 'grapes',
    'kiwi', 'lemon', 'mango', 'orange', 'peach',
    'pear', 'pineapple', 'pomegranate', 'strawberry', 'watermelon'
]
_SORTED_CLASSES = sorted(CLASS_NAMES, key=len, reverse=True)

def extract_label(name: str) -> str:
    low = name.lower()
    for cls in _SORTED_CLASSES:
        if cls in low:
            return cls
    return 'unknown'


def load_dataset(data_dir: Path) -> pd.DataFrame:
    rows = []
    for session_dir in sorted(data_dir.iterdir()):
        if not session_dir.is_dir():
            continue
        voxel_p  = session_dir / 'voxels.json'
        points_p = session_dir / 'points.json'
        graph_p  = session_dir / 'graph.json'
        if not (voxel_p.exists() and points_p.exists() and graph_p.exists()):
            continue
        with open(voxel_p) as f:
            vox = json.load(f)
        with open(points_p) as f:
            pts = json.load(f)
        with open(graph_p) as f:
            grp = json.load(f)
        src_url  = vox.get('obj_url', '')
        src_name = Path(src_url).stem if src_url else session_dir.name
        label = extract_label(src_name)
        if label == 'unknown':
            label = extract_label(session_dir.name)
        rows.append({
            'session': session_dir.name,
            'label':   label,
            'voxels':  np.array(vox['matrix'], dtype=np.float32).reshape(vox['shape']),
            'points':  np.array(pts['points'], dtype=np.float32),
            'nodes':   np.array(grp['nodes'],  dtype=np.float32),
            'edges':   np.array(grp['edges'],  dtype=np.int32),
            'weights': np.array(grp['weights'], dtype=np.float32),
        })
    df = pd.DataFrame(rows)
    df = df[df['label'] != 'unknown'].reset_index(drop=True)
    return df


df = load_dataset(DATA_DIR)
print(f'Loaded {len(df)} labeled samples across {df["label"].nunique()} classes')
print(df['label'].value_counts().to_string())

In [ ]:
DEMO = len(df) < 30
if DEMO:
    print('⚠  DEMO MODE — fewer than 30 samples detected.')
    print('   LOO accuracy will be 0% because each class has only one sample.')
    print('   Run with the full dataset (5,899 samples) for meaningful accuracy metrics.')
else:
    print(f'Full dataset mode — {len(df)} samples.')

print()
print('Voxel grid shape:  ', df['voxels'].iloc[0].shape)
print('Point cloud shape: ', df['points'].iloc[0].shape)
print('Graph nodes/edges: ', len(df['nodes'].iloc[0]), '/', len(df['edges'].iloc[0]))

## D3.2 — Data Visualisation

Three views — one per representation — to confirm the pipeline produces coherent geometry.

In [ ]:
n_show = min(8, len(df))
fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 3))
if n_show == 1:
    axes = [axes]
for ax, (_, row) in zip(axes, df.iterrows()):
    proj = row['voxels'].max(axis=2)
    ax.imshow(proj, cmap='YlOrBr', origin='lower', interpolation='nearest')
    ax.set_title(row['label'], fontsize=9)
    ax.axis('off')
fig.suptitle('Voxel Max-Projection (top-down, 32³ grid)', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
n_show = min(6, len(df))
fig = plt.figure(figsize=(4 * n_show, 4))
for i, (_, row) in enumerate(df.head(n_show).iterrows()):
    ax = fig.add_subplot(1, n_show, i + 1, projection='3d')
    pts = row['points'][::4]
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2],
               c=pts[:, 1], cmap='plasma', s=1.5, alpha=0.7)
    ax.set_title(row['label'], fontsize=9)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
fig.suptitle('Point Cloud (1024 pts, height-coloured)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
labels = df['label'].tolist()
node_counts  = [len(r) for r in df['nodes']]
edge_counts  = [len(r) for r in df['edges']]
mean_degrees = [2 * ec / max(nc, 1) for nc, ec in zip(node_counts, edge_counts)]
for ax, values, title, color in zip(
    axes,
    [node_counts, edge_counts, mean_degrees],
    ['Node Count', 'Edge Count', 'Mean Degree'],
    ['#4e9af1', '#f1a44e', '#4ef18b']
):
    ax.bar(labels, values, color=color, alpha=0.8)
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
fig.suptitle('Graph Topology Statistics per Sample', fontsize=11)
plt.tight_layout()
plt.show()

## D3.3 — Iteration 1: 3D Voxel CNN

### Problem & Representation

**Input:** Each mesh is rasterised into a 32×32×32 binary occupancy grid.  
**Task:** 15-class classification — given the voxel tensor, predict the fruit category.  
**Why voxels:** Voxelisation provides a regular, grid-aligned representation amenable to 3D convolutions, directly analogous to pixels in 2D image classification.  
**Variable-size problem:** Raw meshes have wildly different vertex counts. Voxelisation at a fixed grid resolution normalises geometry into a fixed-size tensor regardless of mesh density.

### Feature Engineering

The 32³ = 32,768-dimensional binary vector is projected to a lower-dimensional space using PCA before feeding into a classifier. This reduces overfitting risk and makes training tractable with small datasets.

In [ ]:
print("""
3D CNN Architecture (full-dataset target)
==========================================
Input: (B, 1, 32, 32, 32)
  |
  +- Conv3d(1->32, k=3, pad=1) -> BN -> ReLU
  +- Conv3d(32->32, k=3, pad=1) -> BN -> ReLU
  +- MaxPool3d(2) -> (B, 32, 16, 16, 16)
  |
  +- Conv3d(32->64, k=3, pad=1) -> BN -> ReLU
  +- Conv3d(64->64, k=3, pad=1) -> BN -> ReLU
  +- MaxPool3d(2) -> (B, 64, 8, 8, 8)
  |
  +- Conv3d(64->128, k=3, pad=1) -> BN -> ReLU
  +- AdaptiveAvgPool3d(2) -> (B, 128, 2, 2, 2)
  |
  +- Flatten -> (B, 1024)
  +- Linear(1024->256) -> ReLU -> Dropout(0.5)
  +- Linear(256->64) -> ReLU
  +- Linear(64->15) -> softmax

Demo substitute: PCA(n=min(N-1, 64)) -> SVM(RBF)
""")

In [ ]:
X_vox = np.stack([row.flatten() for row in df['voxels']])
n_components_max = min(len(df) - 1, 64)

if n_components_max >= 2:
    pca_diag = PCA(n_components=n_components_max).fit(X_vox)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(range(1, n_components_max + 1),
                pca_diag.explained_variance_ratio_ * 100, color='#d4a017', alpha=0.8)
    axes[0].set_xlabel('Component'); axes[0].set_ylabel('Variance Explained (%)')
    axes[0].set_title('Per-Component Variance (Voxel PCA)')
    axes[1].plot(range(1, n_components_max + 1),
                 np.cumsum(pca_diag.explained_variance_ratio_) * 100,
                 color='#d4a017', linewidth=2, marker='o', markersize=4)
    axes[1].axhline(90, color='grey', linestyle='--', alpha=0.5, label='90%')
    axes[1].set_xlabel('Components'); axes[1].set_ylabel('Cumulative Variance (%)')
    axes[1].set_title('Cumulative Variance (Voxel PCA)')
    axes[1].legend()
    plt.tight_layout(); plt.show()
else:
    print('Not enough samples for PCA diagnostics (need >= 3).')

In [ ]:
y = df['label'].values
n_comp = max(1, min(n_components_max, len(df) - 1))

clf_vox = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=n_comp)),
    ('svm',    SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced')),
])

loo = LeaveOneOut()
y_true_v, y_pred_v = [], []
for train_idx, test_idx in loo.split(X_vox):
    clf_vox.fit(X_vox[train_idx], y[train_idx])
    y_true_v.append(y[test_idx][0])
    y_pred_v.append(clf_vox.predict(X_vox[test_idx])[0])

acc_vox = accuracy_score(y_true_v, y_pred_v)
print(f'Voxel SVM  LOO Accuracy: {acc_vox:.1%}  ({int(acc_vox*len(df))}/{len(df)})')
if DEMO:
    print('(Expected 0% in demo mode.)')

present_classes = sorted(set(y_true_v) | set(y_pred_v))
cm_v = confusion_matrix(y_true_v, y_pred_v, labels=present_classes)
fig, ax = plt.subplots(figsize=(max(5, len(present_classes)), max(4, len(present_classes) - 1)))
ConfusionMatrixDisplay(cm_v, display_labels=present_classes).plot(
    ax=ax, colorbar=True, cmap='Oranges', xticks_rotation=45)
ax.set_title('Iteration 1 — Voxel SVM Confusion Matrix (LOO)', fontsize=11)
plt.tight_layout(); plt.show()

## D3.4 — Iteration 2: PointNet-style Point Cloud

### Problem & Representation

**Input:** 1,024 surface points sampled via Farthest Point Sampling (FPS).  
**Task:** Same 15-class classification, operating directly on unordered point sets.  
**Why points:** Point clouds are much more compact than voxels (1024×3 = 3,072 values vs 32,768) and preserve fine surface geometry without voxelisation artifacts.  
**Variable-size problem:** FPS guarantees exactly 1,024 points regardless of the original mesh density.

### Feature Engineering

29 hand-crafted geometric descriptors capturing global shape statistics, PCA eigenvalue ratios, bounding box proportions, and average nearest-neighbour distances.

In [ ]:
print("""
PointNet Architecture (full-dataset target)
============================================
Input: (B, 3, 1024)   <- raw xyz, permutation-invariant
  |
  +- T-Net(3->3)  Input Transform
  +- SharedMLP(3->64->64)
  |
  +- T-Net(64->64)  Feature Transform
  +- SharedMLP(64->64->128->1024)
  |
  +- GlobalMaxPool -> (B, 1024)  <- global shape descriptor
  |
  +- Linear(1024->512) -> BN -> ReLU -> Dropout(0.3)
  +- Linear(512->256)  -> BN -> ReLU -> Dropout(0.3)
  +- Linear(256->15)   -> softmax

Demo substitute: hand-crafted 29-dim descriptor -> SVM(RBF)
""")

In [ ]:
FEATURE_NAMES_PT = (
    [f'{stat}_{ax}' for ax in 'xyz' for stat in ('mean','std','min','max','skew','ptp')] +
    ['pca_ev0','pca_ev1','pca_ev2','pca_ratio_01','pca_ratio_02','pca_ratio_12'] +
    ['bb_vol','bb_x','bb_y','bb_z'] +
    ['mean_nn_dist']
)
assert len(FEATURE_NAMES_PT) == 29


def extract_point_features(pts: np.ndarray) -> np.ndarray:
    feats = []
    for ax in range(3):
        col = pts[:, ax]
        feats += [col.mean(), col.std(), col.min(), col.max(),
                  float(scipy_skew(col)), np.ptp(col)]
    pca = PCA(n_components=3).fit(pts)
    ev = pca.explained_variance_
    feats += [ev[0], ev[1], ev[2],
              ev[0]/(ev[1]+1e-10), ev[0]/(ev[2]+1e-10), ev[1]/(ev[2]+1e-10)]
    bb = pts.max(axis=0) - pts.min(axis=0)
    feats += [float(np.prod(bb)), bb[0], bb[1], bb[2]]
    sub = pts[::4] if len(pts) > 256 else pts
    tree = KDTree(sub)
    dists, _ = tree.query(sub, k=2)
    feats.append(float(dists[:, 1].mean()))
    return np.array(feats, dtype=np.float32)


X_pt = np.stack([extract_point_features(row) for row in df['points']])
print(f'Point feature matrix: {X_pt.shape}')

clf_pt = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced')),
])

y_true_p, y_pred_p = [], []
for train_idx, test_idx in loo.split(X_pt):
    clf_pt.fit(X_pt[train_idx], y[train_idx])
    y_true_p.append(y[test_idx][0])
    y_pred_p.append(clf_pt.predict(X_pt[test_idx])[0])

acc_pt = accuracy_score(y_true_p, y_pred_p)
print(f'PointNet-style SVM  LOO Accuracy: {acc_pt:.1%}  ({int(acc_pt*len(df))}/{len(df)})')

present_classes_p = sorted(set(y_true_p) | set(y_pred_p))
cm_p = confusion_matrix(y_true_p, y_pred_p, labels=present_classes_p)
fig, ax = plt.subplots(figsize=(max(5, len(present_classes_p)), max(4, len(present_classes_p) - 1)))
ConfusionMatrixDisplay(cm_p, display_labels=present_classes_p).plot(
    ax=ax, colorbar=True, cmap='Blues', xticks_rotation=45)
ax.set_title('Iteration 2 — PointNet-style SVM Confusion Matrix (LOO)', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

feat_std = X_pt.std(axis=0)
sorted_idx = np.argsort(feat_std)[::-1]
axes[0].bar(range(len(FEATURE_NAMES_PT)),
            feat_std[sorted_idx], color='#4e9af1', alpha=0.85)
axes[0].set_xticks(range(len(FEATURE_NAMES_PT)))
axes[0].set_xticklabels([FEATURE_NAMES_PT[i] for i in sorted_idx],
                         rotation=75, fontsize=7)
axes[0].set_title('Feature Std (proxy for discriminability)')
axes[0].set_ylabel('Std across samples')

if len(df) >= 3:
    X_pt_scaled = StandardScaler().fit_transform(X_pt)
    pca2d = PCA(n_components=2).fit_transform(X_pt_scaled)
    unique_labels = sorted(df['label'].unique())
    cmap = plt.cm.get_cmap('tab20', len(unique_labels))
    for i, lbl in enumerate(unique_labels):
        mask = df['label'] == lbl
        axes[1].scatter(pca2d[mask, 0], pca2d[mask, 1],
                        label=lbl, color=cmap(i), s=80, alpha=0.9)
    axes[1].legend(fontsize=7, ncol=2)
    axes[1].set_title('PCA 2D of Point Features')
    axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')

plt.tight_layout(); plt.show()

## D3.5 — Iteration 3: Mesh Graph (GCN-style)

### Problem & Representation

**Input:** The mesh is treated as a graph G=(V, E) where vertices are 3D points and edges are mesh edges with Euclidean-distance weights.  
**Task:** Same 15-class fruit classification, operating on graph-structured data.  
**Why graphs:** Graphs capture topological connectivity that neither voxels nor point clouds encode — the branching structure of a pineapple or the elongated topology of a banana emerge naturally from the edge structure.  
**Variable-size problem:** A GCN processes the graph directly with readout via global pooling, making it size-agnostic. Our descriptor-based demo extracts scalar statistics summarising graph topology.

### Feature Engineering

16 graph-level scalar features: node/edge counts, degree distribution moments (mean, std, min, max, Q25, Q75), edge weight statistics, and spatial spread per axis.

In [ ]:
print("""
Graph Convolutional Network Architecture (full-dataset target)
===============================================================
Input: G = (V, E)   V in R^(N x 3),  E subset V x V with edge weights
  |
  +- GCNConv(3->64)   -> ReLU
  +- GCNConv(64->128) -> ReLU
  +- GCNConv(128->256) -> ReLU
  |
  +- global_mean_pool -> (B, 256)
  |
  +- Linear(256->128) -> ReLU -> Dropout(0.3)
  +- Linear(128->64)  -> ReLU
  +- Linear(64->15)   -> softmax

Framework: PyTorch Geometric (torch_geometric)
Demo substitute: 16-dim graph statistics vector -> SVM(RBF)
""")

In [ ]:
FEATURE_NAMES_GR = [
    'node_count', 'edge_count', 'avg_degree',
    'degree_mean', 'degree_std', 'degree_min', 'degree_max',
    'degree_q25', 'degree_q75',
    'weight_mean', 'weight_std', 'weight_min', 'weight_max',
    'spatial_x', 'spatial_y', 'spatial_z',
]
assert len(FEATURE_NAMES_GR) == 16


def extract_graph_features(nodes: np.ndarray, edges: np.ndarray,
                            weights: np.ndarray) -> np.ndarray:
    n = len(nodes)
    e = len(edges)
    degree = np.bincount(edges.flatten(), minlength=n).astype(np.float32)
    spatial = nodes.max(axis=0) - nodes.min(axis=0) if n > 0 else np.zeros(3)
    return np.array([
        n, e, 2 * e / max(n, 1),
        degree.mean(), degree.std(), degree.min(), degree.max(),
        float(np.percentile(degree, 25)), float(np.percentile(degree, 75)),
        weights.mean() if len(weights) else 0,
        weights.std()  if len(weights) else 0,
        weights.min()  if len(weights) else 0,
        weights.max()  if len(weights) else 0,
        spatial[0], spatial[1], spatial[2],
    ], dtype=np.float32)


X_gr = np.stack([
    extract_graph_features(row['nodes'], row['edges'], row['weights'])
    for _, row in df.iterrows()
])
print(f'Graph feature matrix: {X_gr.shape}')

clf_gr = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced')),
])

y_true_g, y_pred_g = [], []
for train_idx, test_idx in loo.split(X_gr):
    clf_gr.fit(X_gr[train_idx], y[train_idx])
    y_true_g.append(y[test_idx][0])
    y_pred_g.append(clf_gr.predict(X_gr[test_idx])[0])

acc_gr = accuracy_score(y_true_g, y_pred_g)
print(f'GCN-style SVM  LOO Accuracy: {acc_gr:.1%}  ({int(acc_gr*len(df))}/{len(df)})')

present_classes_g = sorted(set(y_true_g) | set(y_pred_g))
cm_g = confusion_matrix(y_true_g, y_pred_g, labels=present_classes_g)
fig, ax = plt.subplots(figsize=(max(5, len(present_classes_g)), max(4, len(present_classes_g) - 1)))
ConfusionMatrixDisplay(cm_g, display_labels=present_classes_g).plot(
    ax=ax, colorbar=True, cmap='Greens', xticks_rotation=45)
ax.set_title('Iteration 3 — GCN-style SVM Confusion Matrix (LOO)', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
X_gr_z = (X_gr - X_gr.mean(axis=0)) / (X_gr.std(axis=0) + 1e-10)
fig, ax = plt.subplots(figsize=(max(8, len(FEATURE_NAMES_GR) * 0.7),
                                  max(3, len(df) * 0.55)))
im = ax.imshow(X_gr_z, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax.set_xticks(range(len(FEATURE_NAMES_GR)))
ax.set_xticklabels(FEATURE_NAMES_GR, rotation=60, ha='right', fontsize=8)
ax.set_yticks(range(len(df)))
ax.set_yticklabels(df['label'].tolist(), fontsize=9)
ax.set_title('Graph Features — Z-score Heatmap', fontsize=11)
plt.colorbar(im, ax=ax, label='Z-score')
plt.tight_layout(); plt.show()

## D3.6 — Comparative Analysis

In [ ]:
results = pd.DataFrame([
    {'Iteration': 'D3.3 Voxel CNN',    'Representation': 'Voxel Grid (32^3)',
     'Classifier': 'PCA+SVM(RBF)', 'Feature Dim': n_comp,  'LOO Accuracy': acc_vox},
    {'Iteration': 'D3.4 PointNet',     'Representation': 'Point Cloud (1024x3)',
     'Classifier': 'SVM(RBF)',     'Feature Dim': 29,      'LOO Accuracy': acc_pt},
    {'Iteration': 'D3.5 GCN',          'Representation': 'Mesh Graph G(V,E)',
     'Classifier': 'SVM(RBF)',     'Feature Dim': 16,      'LOO Accuracy': acc_gr},
])
display_df = results.copy()
display_df['LOO Accuracy'] = display_df['LOO Accuracy'].map('{:.1%}'.format)
print(display_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#d4a017', '#f43f5e', '#10b981']
names  = ['Voxel\nSVM', 'PointNet\nSVM', 'GCN\nSVM']
accs   = [acc_vox, acc_pt, acc_gr]
axes[0].bar(names, [a * 100 for a in accs], color=colors, alpha=0.85)
axes[0].set_ylabel('LOO Accuracy (%)')
axes[0].set_title('Accuracy by Iteration')
axes[0].set_ylim(0, 100)
for xi, acc in enumerate(accs):
    axes[0].text(xi, acc * 100 + 1, f'{acc:.0%}', ha='center', fontsize=10)

raw_dims  = [32768, 3072, 0]
feat_dims = [n_comp, 29, 16]
x = np.arange(len(names))
w = 0.35
axes[1].bar(x - w/2, raw_dims,  w, label='Raw Input Dim', color='#555', alpha=0.7)
axes[1].bar(x + w/2, feat_dims, w, label='Feature Dim',   color=colors, alpha=0.85)
axes[1].set_yscale('log')
axes[1].set_xticks(x); axes[1].set_xticklabels(names)
axes[1].set_ylabel('Dimensionality (log)')
axes[1].set_title('Raw vs Feature Dimensionality')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## D3.7 — Discussion

### Observations

**Voxel CNN (D3.3):** The 32³ occupancy grid captures gross shape but is sparse — most fruits are roughly convex, so voxel density gradients carry limited discriminative power. PCA compression is severe (from 32,768 to ≤ N−1 components), meaning the classifier sees a heavily projected view of shape.

**PointNet-style (D3.4):** Surface point clouds preserve geometric detail without voxelisation artifacts. The 29-dimensional descriptor is compact and interpretable. PCA eigenvalue ratios distinguish elongated fruits (banana, pineapple) from compact ones (apple, cherry). Expected to be the strongest demo-mode baseline because the feature space is rich relative to the number of samples.

**GCN (D3.5):** Graph features capture topological connectivity, uniquely suited to fruits with complex geometry (grapes cluster topology, pineapple leaf crown). However, the scalar statistics here lose spatial information. A full GCN operating on node embeddings within the adjacency structure would exploit this representation fully.

### Limitations

1. **Sample size**: 6 demo samples produce LOO accuracy of 0% by construction. All comparisons become meaningful only on the full 5,899-sample dataset.
2. **No data augmentation**: Rotations, reflections, and scale jitter would dramatically increase effective dataset size for all three representations.
3. **Proxy classifiers**: SVM is not the intended classifier for any iteration. Accuracy numbers from the deep models require PyTorch and GPU training.
4. **Label extraction from filenames**: Filenames with unusual conventions (e.g., `12203_Fruit_v1_L3_model`) produce `unknown` and are dropped silently.

### Next Steps (D4+)

- Train true 3D CNN with PyTorch on the full voxel dataset
- Implement PointNet with T-Net alignment modules
- Migrate to PyTorch Geometric for the GCN iteration
- Add confusion matrix analysis to identify hard pairs (e.g., lemon vs avocado)
- Explore ensemble voting across all three representations